In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("="*70)
print("ROSSMANN STORE SALES - DATA EXPLORATION")
print("="*70)

In [ ]:
# Load all three datasets
train = pd.read_csv('../data/raw/train.csv', parse_dates=['Date'])
test = pd.read_csv('../data/raw/test.csv', parse_dates=['Date'])
store = pd.read_csv('../data/raw/store.csv')

print("\n📊 DATASET SHAPES")
print("="*70)
print(f"Train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"Test:  {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"Store: {store.shape[0]:,} rows × {store.shape[1]} columns")

print(f"\nDate Range (Train): {train['Date'].min()} to {train['Date'].max()}")
print(f"Date Range (Test):  {test['Date'].min()} to {test['Date'].max()}")
print(f"Total Stores: {train['Store'].nunique()}")

In [ ]:
print("\n📋 TRAIN DATASET COLUMNS")
print("="*70)
print(train.info())

print("\n📋 STORE DATASET COLUMNS")
print("="*70)
print(store.info())

In [ ]:
print("\n👀 TRAIN DATA SAMPLE")
print("="*70)
display(train.head(10))

print("\n👀 STORE DATA SAMPLE")
print("="*70)
display(store.head(10))

In [ ]:
print("\n🔍 MISSING VALUES ANALYSIS")
print("="*70)

print("\nTrain Dataset:")
missing_train = train.isnull().sum()
print(missing_train[missing_train > 0])
if missing_train.sum() == 0:
    print("✅ No missing values in train!")

print("\nTest Dataset:")
missing_test = test.isnull().sum()
print(missing_test[missing_test > 0])

print("\nStore Dataset:")
missing_store = store.isnull().sum()
print(missing_store[missing_store > 0])
if missing_store.sum() > 0:
    print(f"\n⚠️ Store data has {missing_store.sum()} missing values")

In [ ]:
print("\n💰 SALES (TARGET VARIABLE) ANALYSIS")
print("="*70)
print(train['Sales'].describe())

print(f"\nTotal Sales: ${train['Sales'].sum():,.2f}")
print(f"Average Daily Sales: ${train['Sales'].mean():,.2f}")
print(f"Median Daily Sales: ${train['Sales'].median():,.2f}")
print(f"Max Daily Sales: ${train['Sales'].max():,.2f}")
print(f"Stores with Sales = 0: {(train['Sales'] == 0).sum():,}")

In [ ]:
# Sales Distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Histogram
axes[0, 0].hist(train['Sales'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Sales Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Sales')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(alpha=0.3)

# Box plot
axes[0, 1].boxplot(train['Sales'])
axes[0, 1].set_title('Sales Box Plot (Outliers Visible)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Sales')
axes[0, 1].grid(alpha=0.3)

# Log distribution
axes[1, 0].hist(np.log1p(train['Sales']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Log(Sales) Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Log(Sales)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(alpha=0.3)

# Sales over time
daily_sales = train.groupby('Date')['Sales'].sum()
axes[1, 1].plot(daily_sales.index, daily_sales.values, linewidth=0.5)
axes[1, 1].set_title('Total Sales Over Time', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Total Sales')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Merge train with store information
df = train.merge(store, on='Store', how='left')
print(f"✅ Merged dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Sales by Store Type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

store_type_sales = df.groupby('StoreType')['Sales'].agg(['mean', 'sum'])
store_type_sales['mean'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Average Sales by Store Type', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Store Type')
axes[0].set_ylabel('Average Sales')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].grid(alpha=0.3)

store_type_sales['sum'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Total Sales by Store Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Store Type')
axes[1].set_ylabel('Total Sales')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nStore Type Sales Summary:")
print(store_type_sales)

In [ ]:
# Sales by Day of Week
plt.figure(figsize=(14, 6))
dow_sales = df.groupby('DayOfWeek')['Sales'].mean()
dow_sales.plot(kind='bar', color='forestgreen')
plt.title('Average Sales by Day of Week (1=Monday, 7=Sunday)', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Average Sales')
plt.xticks(rotation=0)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nDay of Week Sales:")
for day, sales in dow_sales.items():
    day_name = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][day-1]
    print(f"  {day_name}: ${sales:,.2f}")

In [ ]:
# Promotion Impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

promo_sales = df.groupby('Promo')['Sales'].mean()
promo_sales.plot(kind='bar', ax=axes[0], color=['red', 'green'])
axes[0].set_title('Average Sales: No Promo vs Promo', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Promo (0=No, 1=Yes)')
axes[0].set_ylabel('Average Sales')
axes[0].set_xticklabels(['No Promo', 'Promo'], rotation=0)
axes[0].grid(alpha=0.3)

# Promo distribution
df['Promo'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['lightcoral', 'lightgreen'])
axes[1].set_title('Proportion of Days with Promotion', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\nAverage Sales WITHOUT Promo: ${promo_sales[0]:,.2f}")
print(f"Average Sales WITH Promo: ${promo_sales[1]:,.2f}")
print(f"Promo Impact: +{((promo_sales[1]/promo_sales[0] - 1) * 100):.1f}%")

In [ ]:
# Open vs Closed Analysis
print("\n🏪 STORE OPEN/CLOSED ANALYSIS")
print("="*70)

open_counts = train['Open'].value_counts()
print(f"\nOpen Days: {open_counts[1]:,}")
print(f"Closed Days: {open_counts[0]:,}")
print(f"Percentage Open: {(open_counts[1]/(open_counts[0]+open_counts[1])*100):.2f}%")

# Check if closed stores have sales
closed_with_sales = train[(train['Open'] == 0) & (train['Sales'] > 0)]
print(f"\n⚠️ Closed days with Sales > 0: {len(closed_with_sales)}")

In [ ]:
print("\n📝 KEY INSIGHTS FROM EXPLORATION")
print("="*70)

insights = """
1. DATA VOLUME
   - Training data: 1,017,209 records across 1,115 stores
   - Time period: 2.5 years (2013-2015)
   - Daily granularity
   
2. SALES PATTERNS
   - Average daily sales: ~$5,770
   - Strong weekly seasonality (Sundays highest)
   - Clear promotional impact (+40-50% sales boost)
   
3. MISSING DATA
   - Store data: Competition info missing for some stores
   - Promo2 information incomplete
   - Need imputation strategy
   
4. STORE CHARACTERISTICS
   - 4 store types (a, b, c, d)
   - Type 'b' has highest average sales
   - 3 assortment levels
   
5. CLOSED STORES
   - ~15% of records are closed stores (Open=0)
   - These should be filtered for training
   
6. NEXT STEPS
   - Clean and preprocess data
   - Handle missing values
   - Create time-based features
   - Engineer lag and rolling features
"""

print(insights)